# Projet FinanceCore : Ingénierie des Données Bancaires
## Étape 1 : Importation des bibliothèques et Connexion

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text, MetaData, Table
from sqlalchemy.dialects.postgresql import insert
from dotenv import load_dotenv

load_dotenv(dotenv_path="t.env")

engine_url = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(engine_url)
df = pd.read_csv(r"C:\Users\elara\Desktop\Dashboard s Multi-Pages\data\financecore_clean.csv")

## Étape 2 : Création du Schéma Relationnel (3NF)

In [ ]:
create_tables_sql = """
DROP VIEW IF EXISTS vue_transactions_details CASCADE;
DROP VIEW IF EXISTS vue_kpi_agence_mois CASCADE;
DROP TABLE IF EXISTS Transaction CASCADE;
DROP TABLE IF EXISTS Compte CASCADE;
DROP TABLE IF EXISTS Client CASCADE;
DROP TABLE IF EXISTS Segment CASCADE;
DROP TABLE IF EXISTS Agence CASCADE;
DROP TABLE IF EXISTS Produit CASCADE;

CREATE TABLE Segment (
    segment_id SERIAL PRIMARY KEY,
    nom_segment VARCHAR(50) UNIQUE NOT NULL
);

CREATE TABLE Client (
    client_id VARCHAR(20) PRIMARY KEY,
    score_credit_client NUMERIC,
    segment_id INT REFERENCES Segment(segment_id)
);

CREATE TABLE Agence (
    agence_id SERIAL PRIMARY KEY,
    nom_agence VARCHAR(100) UNIQUE NOT NULL
);

CREATE TABLE Produit (
    produit_id SERIAL PRIMARY KEY,
    nom_produit VARCHAR(100) UNIQUE NOT NULL
);

CREATE TABLE Compte (
    compte_id SERIAL PRIMARY KEY,
    client_id VARCHAR(20) REFERENCES Client(client_id) ON DELETE CASCADE,
    agence_id INT REFERENCES Agence(agence_id),
    produit_id INT REFERENCES Produit(produit_id),
    categorie_risque VARCHAR(50),
    UNIQUE (client_id, agence_id, produit_id)
);

CREATE TABLE Transaction (
    transaction_id VARCHAR(20) PRIMARY KEY,
    compte_id INT REFERENCES Compte(compte_id) ON DELETE CASCADE,
    date_transaction TIMESTAMP NOT NULL,
    montant NUMERIC NOT NULL,
    montant_eur NUMERIC NOT NULL,
    type_operation VARCHAR(20) CHECK (type_operation IN ('Credit', 'Debit')),
    statut VARCHAR(20),
    solde_avant NUMERIC,
    is_anomalie BOOLEAN
);

CREATE OR REPLACE VIEW vue_transactions_details AS
SELECT t.transaction_id, c.client_id, a.nom_agence, p.nom_produit, t.montant_eur, t.date_transaction
FROM Transaction t
JOIN Compte cp ON t.compte_id = cp.compte_id
JOIN Client c ON cp.client_id = c.client_id
JOIN Agence a ON cp.agence_id = a.agence_id
JOIN Produit p ON cp.produit_id = p.produit_id;

CREATE OR REPLACE VIEW vue_kpi_agence_mois AS
SELECT a.nom_agence, EXTRACT(MONTH FROM t.date_transaction) AS mois,
       COUNT(t.transaction_id) as nb_transactions, SUM(t.montant_eur) AS total_montant_eur
FROM Transaction t
JOIN Compte cp ON t.compte_id = cp.compte_id
JOIN Agence a ON cp.agence_id = a.agence_id
GROUP BY a.nom_agence, EXTRACT(MONTH FROM t.date_transaction);
"""

with engine.begin() as conn:
    conn.execute(text(create_tables_sql))

In [ ]:

#  function correcte avec ON CONFLICT
def insert_on_conflict_nothing(table_name, engine, df, conflict_cols):
    meta = MetaData()
    table = Table(table_name, meta, autoload_with=engine)

    with engine.begin() as conn:
        for _, row in df.iterrows():
            stmt = insert(table).values(**row.to_dict())
            stmt = stmt.on_conflict_do_nothing(index_elements=conflict_cols)
            conn.execute(stmt)

# =========================
#  1. SEGMENT
segments = df[['segment_client']].drop_duplicates()
segments = segments.rename(columns={'segment_client': 'nom_segment'})

insert_on_conflict_nothing('segment', engine, segments, ['nom_segment'])

# =========================
#  2. AGENCE
agences = df[['agence']].drop_duplicates()
agences = agences.rename(columns={'agence': 'nom_agence'})

insert_on_conflict_nothing('agence', engine, agences, ['nom_agence'])

# =========================
#  3. PRODUIT
produits = df[['produit']].drop_duplicates()
produits = produits.rename(columns={'produit': 'nom_produit'})

insert_on_conflict_nothing('produit', engine, produits, ['nom_produit'])

# =========================
#  Reload tables
db_segments = pd.read_sql_table('segment', engine)
db_agences = pd.read_sql_table('agence', engine)
db_produits = pd.read_sql_table('produit', engine)

# =========================
#  4. CLIENT
df_clients = df[['client_id', 'score_credit_client', 'segment_client']].drop_duplicates(subset=['client_id'])

df_clients = df_clients.merge(
    db_segments,
    left_on='segment_client',
    right_on='nom_segment',
    how='left'
)

clients = df_clients[['client_id', 'score_credit_client', 'segment_id']]

insert_on_conflict_nothing('client', engine, clients, ['client_id'])

# =========================
#  5. COMPTE
df_comptes = df[['client_id', 'agence', 'produit']].drop_duplicates()

df_comptes = df_comptes.merge(
    db_agences,
    left_on='agence',
    right_on='nom_agence',
    how='left'
)

df_comptes = df_comptes.merge(
    db_produits,
    left_on='produit',
    right_on='nom_produit',
    how='left'
)

comptes = df_comptes[['client_id', 'agence_id', 'produit_id']]

insert_on_conflict_nothing('compte', engine, comptes, ['client_id', 'agence_id', 'produit_id'])

In [ ]:
#  Charger comptes
db_comptes = pd.read_sql_table('compte', engine)
#  Copier df
df_trans = df.copy()
#  Ajouter agence_id
df_trans = df_trans.merge(
    db_agences,
    left_on='agence',
    right_on='nom_agence',
    how='left'
)
#  Ajouter produit_id
df_trans = df_trans.merge(
    db_produits,
    left_on='produit',
    right_on='nom_produit',
    how='left'
)
#  Lier au compte
df_trans = df_trans.merge(
    db_comptes,
    on=['client_id', 'agence_id', 'produit_id'],
    how='left'
)
df_trans = df_trans.dropna(subset=['compte_id'])

# duplication
df_trans = df_trans.drop_duplicates(subset=['transaction_id'])
# sélectionner colonnes
transactions = df_trans[[
    'transaction_id',
    'compte_id',
    'date_transaction',
    'montant',
    'montant_eur',
    'type_operation',
    'statut',
    'solde_avant',
    'is_anomalie'
]]
#  inserter
insert_on_conflict_nothing(
    'transaction',
    engine,
    transactions,
    ['transaction_id']   
)

## indicateurs simples (KPIs)

In [ ]:
with engine.connect() as conn:
    nb_clients = conn.execute(text("SELECT COUNT(*) FROM Client")).scalar()
    nb_comptes = conn.execute(text("SELECT COUNT(*) FROM Compte")).scalar()
    nb_transactions = conn.execute(text("SELECT COUNT(*) FROM Transaction")).scalar()

print(f"Total Clients : {nb_clients}")
print(f"Total Comptes : {nb_comptes}")
print(f"Total Transactions : {nb_transactions}")


Total Clients : 150
Total Comptes : 1012
Total Transactions : 2000


## transactions pour chaque catégorie de risque

In [ ]:
query_anomalie = """
SELECT cp.categorie_risque, COUNT(t.transaction_id) as total_transactions,
       SUM(CASE WHEN t.is_anomalie = TRUE THEN 1 ELSE 0 END) as nb_anomalies,
       ROUND(SUM(CASE WHEN t.is_anomalie = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(t.transaction_id), 2) AS taux_anomalie_pct
FROM Compte cp
JOIN Transaction t ON cp.compte_id = t.compte_id
GROUP BY cp.categorie_risque
ORDER BY taux_anomalie_pct DESC;
"""
display(pd.read_sql_query(text(query_anomalie), engine))

,categorie_risque,total_transactions,nb_anomalies,taux_anomalie_pct
0,None,2000,313,15.65
